# 1 · Quickstart

`duq` gives you physical **d**imensions, **u**nits and **q**uantities with the *same* semantics on plain Python scalars, NumPy arrays and JAX arrays. `import duq` imports no array library, so it stays fast.

## A quantity

Units are kept **as entered** and arithmetic is dimension-checked.

In [1]:
import duq

bond = duq.Quantity(1.0, "kJ/mol")
bond

Quantity(1.0, Unit('kJ.mol^-1'))

In [2]:
bond.to("eV/mol")

Quantity(6.241509074460763e+21, Unit('eV.mol^-1'))

In [3]:
speed = duq.Quantity(2.0, "m") / duq.Quantity(4.0, "s")
speed

Quantity(0.5, Unit('m.s^-1'))

Mixing incompatible dimensions fails **loud** instead of guessing:

In [4]:
duq.Quantity(1.0, "m") + duq.Quantity(1.0, "s")

DimensionalityError: operands must share a dimension: Dimension.parse('L') vs Dimension.parse('T')

## Namespaces & constants

`duq.units` / `duq.dims` resolve by attribute; CODATA-2022 constants are `Quantity` objects.

In [5]:
print(duq.units.kJ)
print(duq.dims.energy)
print(duq.constants.k_B)
print(duq.constants.N_A.value)

kJ
M·L²·T⁻²
1.380649e-23 J·K⁻¹
6.02214076e+23


## The same quantity on NumPy and JAX

Multiply an array by a unit; units ride through ufuncs and reductions.

In [6]:
import numpy as np

d = np.array([1.0, 2.0, 3.0]) * duq.units.km
print(d.to("m").value)
print(np.sqrt(d * d).unit)

[1000. 2000. 3000.]
km


With the `duq[jax]` extra the same quantity is safe under `grad` — and the gradient is correctly labelled `unit_out / unit_in`.

In [7]:
import duq.jax

def kinetic(v):
    return 0.5 * duq.jax.Quantity(2.0, "kg") * v * v

g = duq.jax.grad(kinetic)(duq.jax.Quantity(3.0, "m/s"))
print(g.value, g.unit)
print(g.unit == duq.unit("J") / duq.unit("m/s"))

6.0 kg·m·s⁻¹
True
